In [85]:
import torch
import torch.nn as nn
import numpy as np

In [5]:
model = nn.LSTMCell(20, 100)
print(model.weight_hh.shape,model.weight_ih.shape)

torch.Size([400, 100]) torch.Size([400, 20])


## Single LSTM CELL

In [25]:
def sigmoid(x):
    return 1 / (1+np.exp(-x))

def lstm_cell(x, h, c, W_hh, W_ih, b):
    i,f,g,o = np.split(W_hh @ h + W_ih @ x + b, 4)
    i,f,g,o = sigmoid(i),sigmoid(f),np.tanh(g),sigmoid(o)
#     print(i.shape,f.shape,g.shape,o.shape)
    c_out = c*f + i*g
    h_out = np.tanh(c_out)*o
    return h_out, c_out

In [10]:
x = np.random.randn(1,20).astype(np.float32)
h0 = np.random.randn(1,100).astype(np.float32)
c0 = np.random.randn(1,100).astype(np.float32)

h_, c_ = model(torch.tensor(x), (torch.tensor(h0), torch.tensor(c0))) # as ref

In [12]:
h_.shape,c_.shape

(torch.Size([1, 100]), torch.Size([1, 100]))

In [26]:
h, c = lstm_cell(x[0], h0[0], c0[0],
                model.weight_hh.detach().numpy(),
                model.weight_ih.detach().numpy(),
                (model.bias_hh + model.bias_ih).detach().numpy())


In [28]:
np.linalg.norm(c - c_[0].detach().numpy())

np.float32(4.4934166e-07)

## Full sequence LSTM

In [54]:
model = nn.LSTM(20, 100, num_layers=1)

In [55]:
X = np.random.randn(50,20).astype(np.float32) # 50（T）是 sequence 长度，20（d） 是 input 的 size
h0 = np.random.randn(1,100).astype(np.float32) 
c0 = np.random.randn(1,100).astype(np.float32)


In [56]:
def lstm(X, h, c, W_hh, W_ih, b):
    H = np.zeros((X.shape[0], h.shape[0]))
    for t in range(X.shape[0]):
        h, c = lstm_cell(X[t], h, c, W_hh, W_ih, b)
        H[t] = h
    return H, c

In [57]:
H, cn = lstm(X, h0[0], c0[0],
            model.weight_hh_l0.detach().numpy(),
            model.weight_ih_l0.detach().numpy(),
            (model.bias_hh_l0 + model.bias_ih_l0).detach().numpy())


In [58]:
H_, (hh_, cn_) = model(torch.tensor(X)[:,None,:],
                       (torch.tensor(h0)[:,None,:],torch.tensor(c0)[:,None,:]))

In [59]:
np.linalg.norm(H - H_[:,0,:].detach().numpy())

np.float64(1.6094598164495483e-06)

## Batching

如果用这种布局， `X[B, T, n]`, 某一个时间步t的数据 `X[B, t, n]` 在 memory 上不连续。

所以 LSTM 中我们如此安排 dimension，`X[T, B, n]`这样某一个时间步的数据 `X[t,B,n]` 在 memory 上是连续的。

In [ ]:
X[T,B,n] # pytorch： (L,N,H_in）

In [60]:
def lstm_cell(x, h, c, W_hh, W_ih, b):
    """
    处理一个 cell（一个时间步）时，x 是 （B，n），axis=0 对应 batch，axis=1对应 hidden size
    """
    i,f,g,o = np.split(h @ W_hh  + x @ W_ih + b, 4, axis=1)   # （b,?)
    i,f,g,o = sigmoid(i),sigmoid(f),np.tanh(g),sigmoid(o)
#     print(i.shape,f.shape,g.shape,o.shape)
    c_out = c*f + i*g
    h_out = np.tanh(c_out)*o
    return h_out, c_out

def lstm(X, h, c, W_hh, W_ih, b):
    H = np.zeros((X.shape[0],X.shape[1], h.shape[1])) # for t=0
    for t in range(X.shape[0]):
        h, c = lstm_cell(X[t], h, c, W_hh, W_ih, b)
        H[t] = h
    return H, c 

In [82]:
X = np.random.randn(50, 128, 20).astype(np.float32) # 50（T）是 sequence 长度，20（d） 是 input 的 size
h0 = np.random.randn(1, 128, 100).astype(np.float32) 
c0 = np.random.randn(1, 128, 100).astype(np.float32)


# ref result by torch code
H_, (hh_, cn_) = model(torch.tensor(X),
                       (torch.tensor(h0),torch.tensor(c0)))



# our own version
H, cn = lstm(X, h0[0], c0[0],
            model.weight_hh_l0.detach().numpy().T,
            model.weight_ih_l0.detach().numpy().T,
            (model.bias_hh_l0 + model.bias_ih_l0).detach().numpy())


In [83]:
H.shape, H_.shape

((50, 128, 100), torch.Size([50, 128, 100]))

In [86]:
np.linalg.norm(H - H_.detach().numpy())

np.float64(1.0371927559567116e-05)

## Training LSTMS

In [ ]:
opt = optim.SGD([W_hh, W_ih, b])

def train_lstm(X, h0, c0, Y, W_hh, W_ih, b, opt)
    H, cn = lstm(X, h0, c0, W_hh, W_ih, b)
    l = loss(H, Y)
    l.backward()
    opt.step()
    
def train_deep_lstm(X, h0, c0, Y, W_hh, W_ih, b, opt):
    H = X
    depth = len(W_hh)
    for d in range(depth):
        H, cn = lstm(H, h0[d], c0[d], W_hh[d], W_ih[d], b[d])
        # truncating and hiddenn unit repackaging
        h0[d] = H[-1].detach().copy()
        c0[d] = cn.detach().copy()
    l = loss(H, Y)
    l.backward()
    opt.step()
    return h0, c0

h0,c0 = zero(...)

for i in range(sequence_len // block_size):
    ho, c0 = train_deep_lstm(X[i*block_size:(i+1)*block_siz], 
                             h0, c0,
                             Y[i*block_size:(i+1)*block_siz], 
                             W_hh, W_ih, b, opt)
    